In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
import scipy.signal
from typing import List, Callable, Iterable, Tuple

from gridops_multidim import BSplineInterpolationAxis, BSplineInterpolationGrid
from gridops_multidim import set_up_grid_axis

from gridops_multidim import multi_inds_from_individual_axes_inds, \
    arbitrary_dim_outer
from gridops_multidim import create_anterpolation_operator
from gridops_multidim import create_restriction_operator_1d, \
    create_restriction_operator
from gridops_multidim import create_prolongation_operator_1d, create_prolongation_operator
from gridops_multidim import create_interaction_operator
from gridops_multidim import create_compute_U_oneplus, \
    create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


In [2]:
%matplotlib notebook

# Basic settings

In [3]:
# geometry
length = 10.0
ndim = 3

# MSM
max_gridlevel = 4

# splines
p = 6
order = p - 1

# particles
n_particles = 5

In [4]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

# Create particle configuration

In [5]:
# TODO: change back to more particles and random charges
#  (few particles and hard-coded charges serve visualization purposes only)

rng = onp.random.default_rng(1632794)
pos = rng.uniform(0., length, size=(n_particles, ndim))
# chg = rng.uniform(-1., 1., size=n_particles)
chg = onp.array([-2, -2, 1, 1, 1])

# Construct grids

In [6]:
grid_axes_all_levels = [None]  # there is no grid at level zero
for lvl in range(1, max_gridlevel + 1):
    h = length / (2 ** (max_gridlevel - lvl))
    print(lvl, h)
    grid_axis = set_up_grid_axis(length=length, h=h, p=p, J_zeroplus=J_zeroplus, periodic=False)
    grid_axes_all_levels.append(grid_axis)
    
# For now, we are just replicating the same axis along all dimensions
# (i.e., same grid spacing, box size, and boundary conditions along all dimensions)
grids_all_levels = [(None, ) * ndim] + [BSplineInterpolationGrid((ga,) * ndim) for ga in grid_axes_all_levels[1:]]
grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

1 1.25
2 2.5
3 5.0
4 10.0


# Construct kernel stencils

In [7]:
sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmjax.kernels import SofteningFunctionOneOverR, split_one_over_r_kernel
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels

In [8]:
# We're using equal grid spacings in all directions here
level_one_gridspacing = grids_all_levels[1].axes[0].h
alpha = 2.5
level_zero_cutoff = alpha * level_one_gridspacing

params_oldmsm = {
    "min_pos": onp.array([0.] * grid_level_one.ndim),
    "max_pos": onp.array([a.length for a in grid_level_one.axes]),
    "level_one_gridspacing": level_one_gridspacing,
    "level_zero_cutoff": level_zero_cutoff,
    "max_gridlevel": max_gridlevel,
    "p": p,
    "mu": 3
}

grids_oldmsm = construct_grids_all_levels(
    min_pos=params_oldmsm["min_pos"],
    max_pos=params_oldmsm["max_pos"],
    p=params_oldmsm["p"],
    level_one_gridspacing=params_oldmsm["level_one_gridspacing"],
    max_gridlevel=params_oldmsm["max_gridlevel"],
)
partial_kernels = split_one_over_r_kernel(
    max_level=max_gridlevel,
    level_zero_cutoff=level_zero_cutoff,
    softening_function=SofteningFunctionOneOverR(p),
)
omega, _ = compute_coeffs_withtruncation(p=params_oldmsm["p"],
                                         mu=params_oldmsm["mu"])
omega_zeroplus = omega[len(omega) // 2:]

kernelstencils_old = compute_kernel_stencils_all_gridlevels(
    kernelfunctions=partial_kernels,
    grids=grids_oldmsm,
    level_zero_cutoff=level_zero_cutoff,
    omega_zeroplus=omega_zeroplus,
)
kernelstencils_old_symmetric = [None]
for stncl in kernelstencils_old[1:]:
    pw = [(s - 1, 0) for s in stncl.shape]
    stncl_symm = jnp.pad(stncl, pad_width=pw, mode='reflect')
    kernelstencils_old_symmetric.append(stncl_symm)

# Functions

## Anterpolation

In [9]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [10]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_multiparticle(pos)

248 µs ± 44.5 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_gradient_multiparticle(pos)

314 µs ± 45.8 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
vals, inds = evaluate_bspline_basis_multiparticle(pos)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)

In [13]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(pos)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)
    return inds, vals, grads

In [14]:
jax.device_put(pos)
%timeit combined(pos)

405 µs ± 62.8 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
anterpolate_level_one = create_anterpolation_operator(grid=grid_level_one)
jitted_anterpolate_level_one = jax.jit(anterpolate_level_one)

In [16]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate_level_one(pos, chg).block_until_ready()

426 µs ± 79.9 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
gridcharge_level_one = anterpolate_level_one(pos, chg)

### Test grid charges summing to net particle charge

In [18]:
anterpolation_funcs_all_levels = [None] + [
    jax.jit(create_anterpolation_operator(g)) for g in grids_all_levels[1:]]

gridcharges_direct = [None] + [af(pos, chg) for af in
                               anterpolation_funcs_all_levels[1:]]


In [19]:
net_charge = chg.sum()

for gc in gridcharges_direct[1:]:
    assert jnp.isclose(gc.sum(), net_charge)

### Plot particles and grid charge

In [20]:
flat_indices = jnp.arange(gridcharge_level_one.size)
unraveled_indices = jnp.unravel_index(flat_indices, grid_level_one.shape)
grid_points_individual_axes = []
for i, axis in enumerate(grid_level_one.axes):
    points = axis.to_raw_indices(unraveled_indices[i]) * axis.h
    grid_points_individual_axes.append(points)

In [21]:
mask = onp.abs(gridcharge_level_one.ravel()) >= 0.01

x, y, z, = grid_points_individual_axes

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(x.ravel()[mask], y.ravel()[mask], z.ravel()[mask], c=gridcharge_level_one.ravel()[mask], s=10)
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=chg, s=100)

plt.show()

<IPython.core.display.Javascript object>

## Restriction

In [22]:
restriction_funcs_all_levels = [None, None]
for lvl in range(2, len(grids_all_levels)):
    rf = create_restriction_operator(grid_source_fine=grids_all_levels[lvl - 1],
                                     grid_target_coarse=grids_all_levels[lvl])
    rf = jax.jit(rf)
    restriction_funcs_all_levels.append(rf)

# TODO: make output shapes of calculation via interpolation and restriction compatible
#  (the former currently returns a flat array, the latter a multi-dimensional one)
gridcharges_via_restriction = [None, gridcharges_direct[1].reshape(grids_all_levels[1].shape)]
for lvl in range(2, len(grids_all_levels)):
    rf = restriction_funcs_all_levels[lvl]
    gc_lowergrid = gridcharges_via_restriction[lvl - 1]
    gc = rf(gc_lowergrid)
    gridcharges_via_restriction.append(gc)

In [23]:
for gc_direct, gc_restrict in zip(gridcharges_direct[2:],
                                  gridcharges_via_restriction[2:]):
    assert jnp.allclose(gc_direct, gc_restrict) # TODO: shapes

## Prolongation

In [24]:
prolongate = create_prolongation_operator(
    grid_target_fine=grids_all_levels[1],
    grid_source_coarse=grids_all_levels[2],
)

In [25]:
gridcharges_via_restriction[2].shape

(11, 11, 11)

In [26]:
prolongate(gridcharges_via_restriction[2]).shape

(15, 15, 15)

## Interaction

In [27]:
kernelstencil_level_one = kernelstencils_old_symmetric[1]
gridcharge_in = gridcharge_level_one.reshape(grid_level_one.shape)

@jax.jit
def interact_custom(in_array):
    return create_interaction_operator(grid_level_one, kernelstencil_level_one)(in_array)

@jax.jit
def interact_builtin_conv_jax(in_array):
    return jax.scipy.signal.convolve(in_array, kernelstencil_level_one, mode="same")

def interact_builtin_conv_scipy(in_array):
    return scipy.signal.convolve(in_array, onp.array(kernelstencil_level_one), mode="same")


In [40]:
jax.device_put(gridcharge_in)
%timeit interact_custom(gridcharge_in).block_until_ready()

1.17 ms ± 50.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [29]:
jax.device_put(gridcharge_in)
%timeit interact_builtin_conv_jax(gridcharge_in).block_until_ready()

1.71 ms ± 159 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [39]:
arr = onp.array(gridcharge_in)
%timeit interact_builtin_conv_scipy(arr)

555 µs ± 101 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [31]:
result_custom = interact_custom(gridcharge_in)
result_builtin_conv_jax = interact_builtin_conv_jax(gridcharge_in)
result_builtin_conv_scipy = interact_builtin_conv_scipy(gridcharge_in)

assert jnp.allclose(result_custom, result_builtin_conv_jax)
assert jnp.allclose(result_custom, result_builtin_conv_scipy)

## End-to-end

## Helper functions

In [32]:
@jax.jit
def sum_kernels_1_to_L(r):
    return jnp.sum(jnp.asarray([k(r) for k in partial_kernels[1:]]))


def compute_U_oneplus_exact_loop(positions, charges):
    U_oneplus_direct = 0.
    for i in range(len(positions)):
        for j in range(0, len(positions)):
            r_ij = jnp.linalg.norm(positions[i] - positions[j])
            U_oneplus_direct += charges[i] * charges[
                j] * sum_kernels_1_to_L(r_ij)
    U_oneplus_direct *= 0.5

    return U_oneplus_direct


@jax.jit
def compute_U_oneplus_exact_vectorized(positions, charges):
    r_ij = jnp.abs(positions[:, jnp.newaxis] - positions)
    qi_qj = charges[:, jnp.newaxis] * charges

    return 0.5 * (qi_qj * jax.vmap(jax.vmap(sum_kernels_1_to_L))(r_ij)).sum()

@jax.jit
def compute_U_oneplus_grid(positions, charges):
    return create_compute_U_oneplus(
        grids=grids_all_levels,
        kernel_stencils=kernelstencils_old_symmetric,
    )(positions, charges)

In [33]:
k_prime = jax.jit(jax.grad(sum_kernels_1_to_L))


def compute_f_oneplus_exact_loop(positions, charges):
    f_oneplus_direct = []
    for i in range(len(positions)):
        force = 0.
        for j in range(len(positions)):
            if i == j:
                continue
            x_i_minus_x_j = positions[i] - positions[j]
            r_ij = jnp.abs(x_i_minus_x_j)
            force -= charges[j] * (x_i_minus_x_j / r_ij) * k_prime(r_ij)
        force *= charges[i]
        f_oneplus_direct.append(force)

    return jnp.array(f_oneplus_direct)


@jax.jit
def compute_f_oneplus_exact_vectorized(positions, charges):
    return -jax.grad(compute_U_oneplus_exact_vectorized, argnums=0)(
        positions, charges
    )


@jax.jit
def compute_U_and_f_oneplus_grid(positions, charges):
    return create_compute_U_and_f_oneplus(
        grids=grids_all_levels,
        kernel_stencils=kernelstencils_old_symmetric,
    )(positions, charges)

## Comparison

### Energy

In [34]:
compute_U_oneplus_grid(pos, chg)

Array(4.32159894, dtype=float64)

In [35]:
compute_U_oneplus_exact_loop(pos, chg)

Array(4.46428746, dtype=float64)

In [36]:
compute_U_oneplus_exact_loop(pos, chg)

Array(4.46428746, dtype=float64)

### Forces

In [37]:
# Double-check correctness of exact calculation by comparing different ways of doing it
f_oneplus_exact_loop = compute_f_oneplus_exact_loop(pos, chg)
f_oneplus_exact_vectorized = compute_f_oneplus_exact_vectorized(pos, chg)
f_grad_U_oneplus_exact_vectorized = -jax.grad(
    compute_U_oneplus_exact_vectorized, argnums=0)(pos, chg)
assert jnp.allclose(f_oneplus_exact_loop, f_oneplus_exact_vectorized)
assert jnp.allclose(f_oneplus_exact_loop, f_grad_U_oneplus_exact_vectorized)

In [38]:
# Compare two different ways of computing the grid approximation for the forces
_, f_oneplus_grid = compute_U_and_f_oneplus_grid(pos, chg)
f_grad_U_oneplus_grid = -jax.grad(compute_U_oneplus_grid, argnums=0)(pos, chg)
assert jnp.allclose(f_oneplus_grid, f_grad_U_oneplus_grid)

ValueError: Incompatible shapes for broadcasting: shapes=[(5, 216), (5, 216, 3)]

In [ ]:
# Compare exact and grid forces
fig, ax = plt.subplots()
ax.set_xlabel("forces $f^{1+}$ (exact)")
ax.set_ylabel("forces $f^{1+}$ (grid)")
ax.scatter(f_oneplus_exact_loop, f_oneplus_grid)
xlim = ax.get_xlim()
ylim = ax.get_ylim()
# plot parity line and zero axes
ax.plot(xlim, xlim, color="black", zorder=-1)
ax.axhline(color="black", linewidth=0.75)
ax.axvline(color="black", linewidth=0.75)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()